In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/enron_spam_data.csv")

In [3]:

df.head(10)

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",ham,1999-12-14
6,6,meter 1517 - jan 1999,"george ,\ni need the following done :\njan 13\...",ham,1999-12-14
7,7,duns number changes,fyi\n- - - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
8,8,king ranch,there are two fields of gas that i am having d...,ham,1999-12-14
9,9,re : entex transistion,thanks so much for the memo . i would like to ...,ham,1999-12-14


In [14]:
# Scikit-learn version: TF-IDF features + a one-hidden-layer MLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

sklearn_texts = (df['Subject'].fillna('') + ' ' + df['Message'].fillna('')).astype(str)
sklearn_labels = df['Spam/Ham'].map({'spam': 1, 'ham': 0})
sklearn_x_train, sklearn_x_test, sklearn_y_train, sklearn_y_test = train_test_split(
    sklearn_texts, sklearn_labels, test_size=0.2, random_state=42, stratify=sklearn_labels
)

sklearn_vectorizer = TfidfVectorizer(max_features=10000)
sklearn_x_train_tfidf = sklearn_vectorizer.fit_transform(sklearn_x_train)
sklearn_x_test_tfidf = sklearn_vectorizer.transform(sklearn_x_test)

sklearn_model = MLPClassifier(
    hidden_layer_sizes=(128,), max_iter=100, early_stopping=True, random_state=42
)
sklearn_model.fit(sklearn_x_train_tfidf, sklearn_y_train)
sklearn_predictions = sklearn_model.predict(sklearn_x_test_tfidf)
sklearn_accuracy = accuracy_score(sklearn_y_test, sklearn_predictions)
print(f"Scikit-learn test accuracy: {sklearn_accuracy:.2%}")

Scikit-learn test accuracy: 99.94%


In [ ]:
df['text'] = df['Subject'].fillna('') + ' ' + df['Message'].fillna('')
df['text'] = df['text'].astype('string')
df['label'] = df['Spam/Ham'].map({'spam': 1, 'ham': 0})

In [5]:
print(df['Spam/Ham'].unique())
print(df['label'].isna().sum())
print(df.dtypes)

['ham' 'spam']
0
Unnamed: 0             int64
Subject               object
Message               object
Spam/Ham              object
Date                  object
text          string[python]
label                  int64
dtype: object


In [6]:
# before writing TF IDF just split so the information won't be in test set
np.random.seed(42)
# shuffling the indices for randomness
# without shuffling can be also done
# len(df) # 33716
# we can use stratification as well to split the data to maintain the 
# ratio of labels in both training and test sets
shuffled_indices = np.random.permutation(len(df))
# print(shuffled_indices)
test_size = int(len(df) * 0.2)

test_indices = shuffled_indices[:test_size]
train_indices = shuffled_indices[test_size:]

train_df = df.iloc[train_indices]
test_df = df.iloc[test_indices]

x_train, x_test = train_df['text'], test_df['text']
x_label, y_label = train_df['label'], test_df['label']

print(x_train[:5])
x_label[:5]

9647     fw : memo : re : your work phone number hi ,
i...
17887    start date : 1 / 26 / 02 ; hourahead hour : 19...
28698    interesting article interesting article , espe...
8292     california update 1 / 22 / 01 executive summar...
31315    fw : re ivanhoe e . s . d fyi , kim .
- - - - ...
Name: text, dtype: string


9647     1
17887    0
28698    0
8292     0
31315    1
Name: label, dtype: int64

In [7]:
from collections import Counter
import math

def tokenize(text):
    # simple whitespace + lowercase tokenizer
    return text.lower().split()

def build_vocab(train_texts, max_features=10000):
    doc_freq = Counter()
    for text in train_texts:
        unique_words = set(tokenize(text))
        for word in unique_words:
            doc_freq[word] += 1
    # keep the most common max_features words by document frequency
    most_common = doc_freq.most_common(max_features)
    vocab = {word: idx for idx, (word, _) in enumerate(most_common)}
    return vocab, doc_freq

def compute_idf(vocab, doc_freq, n_docs):
    idf = np.zeros(len(vocab))
    for word, idx in vocab.items():
        # smoothed idf, same formula sklearn uses by default
        idf[idx] = math.log((1 + n_docs) / (1 + doc_freq[word])) + 1
    return idf

def transform(texts, vocab, idf):
    num_docs = len(texts)
    num_vocab = len(vocab)
    matrix = np.zeros((num_docs, num_vocab))

    for i, text in enumerate(texts):
        tokens = tokenize(text)
        terms_counts = Counter(tokens)
        total_terms = len(tokens)

        for word, count in terms_counts.items():
            if word in vocab:
                idx = vocab[word]
                tf = count / total_terms if total_terms > 0 else 0
                matrix[i, idx] = tf * idf[idx]

        # L2 normalize each row (also matches sklearn's default behaviour)
        norm = np.linalg.norm(matrix[i])
        if norm > 0:
            matrix[i] = matrix[i] / norm

    return matrix

# --------------- fit on train------------------
vocab, doc_freq = build_vocab(x_train, max_features=10000)
idf = compute_idf(vocab, doc_freq, n_docs=len(x_train))

# --------------- transform train and test using the same vocab / idf -------
x_train_tfidf = transform(x_train, vocab, idf)
x_test_tfidf = transform(x_test, vocab, idf)

In [12]:
# FFN/ binary classifier using MLP arch
# it's a classic approach

import torch
import torch.nn as nn

vocab_size = len(vocab)
# hidden_size = 2048
hidden_size = 128
out = 1

class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        return x @ self.weights.T + self.bias

class Adam:
    def __init__(self, params, beta1=0.9, beta2=0.999, lr=1e-3, weight_decay=1e-3, eps=1e-8):
        self.params = list(params)
        self.beta1 = beta1
        self.beta2 = beta2
        self.lr = lr
        self.weight_decay = weight_decay
        self.eps = eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            grad = p.grad
            # these add_, mul_ and addcmul_ are in place operations
            # p.data.mul_(1 - self.lr * self.weight_decay)
            self.m[i].mul_(self.beta1).add_(grad, alpha=1 - self.beta1)
            self.v[i].mul_(self.beta2).addcmul_(grad, grad, value=1 - self.beta2)
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            p.data.addcdiv_(m_hat, v_hat.sqrt().add(self.eps), value=-self.lr)

def relu(logits):
    return torch.clamp(logits, min=0)


def sigmoid(logits):
    return 1.0 / (1.0 + torch.exp(-logits))

def loss_calculation(logits, y):
    sig = sigmoid(logits)
    sig = torch.clamp(sig, 1e-7, 1 - 1e-7)
    loss = -(y * torch.log(sig) + ((1 - y) * torch.log(1 - sig)))
    return loss


class SpamFilter(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = Linear(vocab_size, hidden_size)
        self.hidden_layer = Linear(hidden_size, out)

    def forward(self, input, target=None):
        input = self.mlp(input)
        input = relu(input)
        logits = self.hidden_layer(input)
        loss = None
        if target is not None:
            loss = loss_calculation(logits, target)
        return logits, loss

    def predict(self, input):
        logits, _ = self(input, None)
        prob = sigmoid(logits)
        return 1 if prob.item() >= 0.5 else 0

# TF-IDF input for the model
def get_input(index):
    features = torch.tensor(x_train_tfidf[index], dtype=torch.float32)
    label = torch.tensor(x_label.iloc[index], dtype=torch.float32)
    return features, label

def get_tfidf_input(text):
    features = transform([text], vocab, idf)[0]
    return torch.tensor(features, dtype=torch.float32)

# ----------------------------------------------------------------------------

model = SpamFilter()
# adam_optim = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8)
adam_optim = Adam(model.parameters(), lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8)

for step in range(1000):
    index = step % len(x_train_tfidf)
    x, y = get_input(index)
    adam_optim.zero_grad()
    logits, loss = model(x, y)
    loss = loss.mean()
    loss.backward()
    adam_optim.step()
    if step % 100 == 0:
        print(f"step: {step} | loss: {loss:.4f}")

# ----------------------------------------------------------------------------

is_spam = model.predict(get_tfidf_input("you won a bank"))
if is_spam:
    print("Spam")
else:
    print("Not spam")

# ----------------------------------------------------------------------------

step: 0 | loss: 0.6938
step: 100 | loss: 0.5625
step: 200 | loss: 0.0636
step: 300 | loss: 0.1030
step: 400 | loss: 0.0214
step: 500 | loss: 0.0229
step: 600 | loss: 0.0049
step: 700 | loss: 0.0078
step: 800 | loss: 0.0003
step: 900 | loss: 0.0047
Not spam


In [13]:
# Evaluate the trained model on the held-out test data
model.eval()
correct_predictions = 0

with torch.no_grad():
    for features, label in zip(x_test_tfidf, y_label):
        features = torch.tensor(features, dtype=torch.float32)
        logits, _ = model(features)
        prediction = int(sigmoid(logits).item() >= 0.5)
        correct_predictions += prediction == int(label)

test_accuracy = correct_predictions / len(y_label)
print(f"Test accuracy: {test_accuracy:.2%}")

Test accuracy: 99.87%
